In [4]:
from transformers import pipeline

MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
clf = pipeline("sentiment-analysis", model=MODEL)

texts = [
    # 긍정 2 · 부정 2 · 판단이 갈릴 문장 1 — 직접 작성
    
    "I can do well today", "It probably won't go well", "I just feel good", "It's October today", "I don't want to do it"
]
clf(texts)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998456239700317},
 {'label': 'NEGATIVE', 'score': 0.999560534954071},
 {'label': 'POSITIVE', 'score': 0.9998574256896973},
 {'label': 'POSITIVE', 'score': 0.9988956451416016},
 {'label': 'NEGATIVE', 'score': 0.9995680451393127}]

In [6]:
import numpy as np
import pandas as pd
from scipy.special import expit

def predict_proba_from_model(model: dict, df) -> np.ndarray:
    """
    load_model()이 돌려준 dict로 df 각 행의 양성 확률을 계산한다.

    model: feature_names, coefficients, intercept, scaler{mean, scale} 포함.
    df: pandas DataFrame. 열 순서는 feature_names와 달라도 된다.
    반환: shape (len(df),) 확률 배열.
    오류: feature_names 중 df에 없는 열이 있으면, 빠진 열 이름을 담은 ValueError.
    """
    needed_feature = []
    feature_names = model['feature_names']
    for feature_name in feature_names:    
        if feature_name not in df.columns:
            needed_feature.append(feature_name)
    if len(needed_feature) != 0 :
        raise ValueError(f"need feature : {needed_feature}")
    
    sorted_df = df[feature_names]
    for idx, mean in enumerate(model["scaler"]["mean"]):
        sorted_df.iloc[:,idx] -= mean
        sorted_df.iloc[:,idx] /= model["scaler"]["scale"][idx]
    value = sorted_df@model['coefficients'] + model['intercept']
    return expit(value).to_numpy()

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
BASE = "distilbert/distilbert-base-uncased"
import torch

In [ ]:
# week09/pipeline_internals.ipynb — Day 1

def classify_manual(texts: list[str], model_name: str) -> list[dict]:
    """
    pipeline을 쓰지 않고 토크나이저·모델·소프트맥스를 직접 호출해 감성분석 결과를 만든다.

    texts: 분류할 문장 리스트.
    model_name: pipeline이 쓴 것과 동일한 체크포인트 이름.
    반환: 문장별로 {"label": str, "score": float} 딕셔너리의 리스트.
          label은 model.config.id2label에서 가져오고, score는 그 label의 소프트맥스 확률.
    예: classify_manual(["I love it"], name) -> [{"label": "POSITIVE", "score": 0.9998}]
    """
    if texts == []:
        raise ValueError('빈 문자열입니다.')
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    enc = tok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=1).tolist()
    probabilities = probabilities.to(torch.float32).tolist()
    predicted_classes = [model.config.id2label[p_class] for p_class in predicted_class ]
    result = []
    for idx in range(len(predicted_classes)):
        result.append({'label':predicted_classes[idx], 'score':probabilities[idx][int(predicted_class[idx])]})
    return result
classify_manual(texts, MODEL)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998456239700317},
 {'label': 'NEGATIVE', 'score': 0.999560534954071},
 {'label': 'POSITIVE', 'score': 0.9998574256896973},
 {'label': 'POSITIVE', 'score': 0.9988956451416016},
 {'label': 'NEGATIVE', 'score': 0.9995680451393127}]